# Proyecto Python – Análisis Exploratorio de Datos
## Dataset: Productos Bancarios

**Dataset:** `p4ds-dataset-bancario.csv`  
**Objetivo:** Realizar el EDA del dataset bancario, limpiarlo y exportar el resultado a `data/processed/`.

## 1. Carga y Estructura

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs('../data/processed', exist_ok=True)

df = pd.read_csv('../data/raw/p4ds-dataset-bancario.csv', sep=';')
print('Shape:', df.shape)
df.head()

In [ ]:
print('=== Tipos de datos ===')
print(df.dtypes)
print()
print('=== Valores nulos ===')
print(df.isnull().sum())
print()
print('=== Estadísticas descriptivas ===')
df.describe(include='all')

**Observaciones iniciales:**
- `Monto USD` y `Vendedor` tienen 325 valores nulos (corresponden a productos sin monto asignado como Cuenta Sueldo y Tarjeta de Crédito).
- `Fecha Registro en la App` tiene 249 nulos (clientes no registrados en la app).
- `Monto USD` está almacenado como string con coma decimal (formato europeo).

## 2. Análisis Exploratorio de Datos (EDA)

In [ ]:
# Distribución de productos
print('Distribución por Producto:')
print(df['Producto'].value_counts())
print()
print('Distribución por Estado de Mora:')
print(df['estado_mora'].value_counts())
print()
print('Distribución por Estado App:')
print(df['Estado App'].value_counts())
print()
print('Top 10 sucursales:')
print(df['Sucursal'].value_counts().head(10))

In [ ]:
# Visualización exploratoria
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df['Producto'].value_counts().plot(kind='bar', ax=axes[0,0], color='steelblue', title='Cantidad por Producto')
axes[0,0].tick_params(axis='x', rotation=30)

df['estado_mora'].value_counts().plot(kind='bar', ax=axes[0,1], color='coral', title='Estado de Mora')

df['Sucursal'].value_counts().head(10).plot(kind='barh', ax=axes[1,0], color='mediumseagreen', title='Top 10 Sucursales')

df['Estado App'].value_counts().plot(kind='pie', ax=axes[1,1], autopct='%1.1f%%', title='Estado App', colors=['steelblue','lightgray'])
axes[1,1].set_ylabel('')

plt.tight_layout()
plt.show()

## 3. Limpieza de Datos

In [ ]:
df_clean = df.copy()

# --- Fechas ---
df_clean['Fecha'] = pd.to_datetime(df_clean['Fecha'], dayfirst=True, errors='coerce')
df_clean['Fecha Registro en la App'] = pd.to_datetime(df_clean['Fecha Registro en la App'], dayfirst=True, errors='coerce')

# --- Monto USD: coma decimal → punto, convertir a float ---
df_clean['Monto USD'] = (
    df_clean['Monto USD']
    .astype(str)
    .str.replace('.', '', regex=False)   # quitar separador de miles
    .str.replace(',', '.', regex=False)  # coma → punto decimal
    .str.replace('nan', '', regex=False)
)
df_clean['Monto USD'] = pd.to_numeric(df_clean['Monto USD'], errors='coerce')

# --- Rellenar nulos ---
df_clean['Vendedor'] = df_clean['Vendedor'].fillna('Sin vendedor')
df_clean['Monto USD'] = df_clean['Monto USD'].fillna(0.0)

# --- Nueva columna: días hasta registrarse en la app ---
df_clean['Dias_hasta_app'] = (
    df_clean['Fecha Registro en la App'] - df_clean['Fecha']
).dt.days

# --- Nueva columna: tiene mora (booleano) ---
df_clean['Tiene_mora'] = df_clean['estado_mora'].isin(['En Mora', 'Incobrable'])

print('Shape después de limpieza:', df_clean.shape)
print()
print('Nulos restantes:')
print(df_clean.isnull().sum())
df_clean.head()

In [ ]:
# Estadísticas del Monto USD luego de limpieza
montos = df_clean[df_clean['Monto USD'] > 0]['Monto USD']
print('Estadísticas de Monto USD (excluye ceros):')
print(montos.describe())
print(f'Mediana: {montos.median():,.2f}')
print(f'Rango: {montos.max() - montos.min():,.2f}')

In [ ]:
# Monto promedio por producto
df_clean[df_clean['Monto USD'] > 0].groupby('Producto')['Monto USD'].mean().sort_values(ascending=False).plot(
    kind='barh', figsize=(10, 5), color='steelblue', title='Monto USD promedio por Producto'
)
plt.xlabel('Monto USD promedio')
plt.tight_layout()
plt.show()

In [ ]:
# Mora por producto
mora_producto = df_clean.groupby('Producto')['Tiene_mora'].mean().sort_values(ascending=False) * 100
mora_producto.plot(kind='bar', figsize=(10, 5), color='coral', title='% de mora por Producto')
plt.ylabel('% con mora o incobrable')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 4. Grabación del DataFrame procesado

In [ ]:
df_clean.to_csv('../data/processed/bancario_procesado.csv', index=False)
print('✅ Archivo guardado: data/processed/bancario_procesado.csv')
print(f'   Filas: {len(df_clean)} | Columnas: {len(df_clean.columns)}')
print(f'   Columnas: {list(df_clean.columns)}')